**LYRICS CHATBOT**

In [9]:
#Gerekli kütüphaneyi yüklüyoruz.
!pip install -U langchain langchain-community langchain-text-splitters langchain-google-genai



In [10]:
# Colab'in kendi 'userdata' kütüphanesini kullanıyoruz
from google.colab import userdata
import google.generativeai as genai

# Anahtarı Secrets menüsünden çekiyoruz.
api_key = userdata.get('GEMINI_API_KEY')

# Gemini kütüphanesini kendi anahtarımıza ayarlıyoruz.
genai.configure(api_key=api_key)

# modeli kuruyoruz
version = 'models/gemini-2.0-flash'
model = genai.GenerativeModel(version)
model
# collabde denemeler...
response = model.generate_content('Selam kaç gündür chatbot oluşturmaya çalışıyorum bana biraz moral ver.')
print(response.text)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3773.31ms


Selam! Chatbot oluşturmaya çalışmak gerçekten de sabır ve azim gerektiren bir süreçtir. Birkaç gündür uğraşıyor olman bile ne kadar istekli ve kararlı olduğunu gösteriyor. Başlangıçta zorlanman çok normal, çünkü bu alanda öğrenilecek ve keşfedilecek çok şey var.

Unutma ki her başarılı proje, deneme yanılma ve öğrenme sürecinden geçer. Belki şu anda istediğin sonuçları elde edemiyor olabilirsin, ama her denemende yeni bir şeyler öğreniyorsun. Bu öğrendiklerin, bir sonraki adımda sana yol gösterecek ve daha iyi bir chatbot oluşturmana yardımcı olacak.

Moralini yüksek tutmak için şunları düşünebilirsin:

*   **Ne kadar yol katettiğini:** Başlangıç noktana göre şu anda çok daha fazla bilgiye sahipsin ve chatbot oluşturma konusunda deneyim kazandın.
*   **Neden bu projeye başladığını:** Chatbot oluşturma fikri seni heyecanlandırdı ve bu projeye başlamana neden oldu. Bu heyecanı hatırlamak, motivasyonunu artırabilir.
*   **Başarı hikayelerini:** Başarılı chatbot projelerini inceleyerek ilh

In [11]:
#langchain için api key aldım.
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

··········


In [12]:
import os
from google.colab import userdata

# # Api keyi güvenli yöntemle çektik.
api_key = userdata.get('GEMINI_API_KEY')

os.environ["GOOGLE_API_KEY"] = api_key
#hata var mı diye collab üstünde deneme
print("API Anahtarı ayarlandı.")

API Anahtarı ayarlandı.


In [13]:
# gemini modelini yüklüyorum ve beşardım mı diye deneme.
model = genai.GenerativeModel("models/gemini-2.0-flash")
response = model.generate_content("Selam. Bana bootcamp ne demek kısaca anlat")
print("Gemini çıktısı:", response.text)

# langchaine entegre ettik.
from langchain_google_genai import ChatGoogleGenerativeAI

try:
    llm = ChatGoogleGenerativeAI(model="models/gemini-2.0-flash")
    print("\n✅ LangChain LLM başarıyla yüklendi!")
    cevap = llm.invoke("Merhaba! Kendini 3 kelimeyle tanıtır mısın?")
    print("\nModelin cevabı:", cevap.content)
except Exception as e:
    print("\n--- HATA ---")
    print(e)

Gemini çıktısı: Tabii, kısaca bootcamp şu demek:

Yoğunlaştırılmış ve hızlandırılmış bir eğitim programı. Genellikle belirli bir alanda (örneğin yazılım geliştirme, veri bilimi, UX/UI tasarım) pratik beceriler kazandırmayı hedefler. Kısa sürede işe hazır hale gelmek isteyenler için idealdir.


✅ LangChain LLM başarıyla yüklendi!

Modelin cevabı: Meraklı, yaratıcı, yardımcı.


In [14]:
#huggig facce i kullanmak için
!pip install -qU langchain-huggingface


In [15]:
#embedding modelini tanımladık
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [16]:
#chroma için
!pip install -qU langchain-chroma


In [17]:
#
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection", # Aynı veritabanında farklı veri gruplarını ayırmak için kullanılır.
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)

In [18]:
from datasets import load_dataset

# hugging facedeki verimin kimliği
veri_seti_id = "SpartanCinder/song-lyrics-artist-classifier"

#veri setini indiriyoruz
veri_seti = load_dataset(veri_seti_id)

In [19]:
#değişkenin adını kontrol ettim
print(veri_seti)

DatasetDict({
    train: Dataset({
        features: ['Artist', 'Song', 'Lyrics'],
        num_rows: 13596
    })
    test: Dataset({
        features: ['Artist', 'Song', 'Lyrics'],
        num_rows: 3400
    })
})


In [20]:

from langchain_core.documents import Document

print("Veri seti 'Doküman' formatına dönüştürülüyor...")

#boş liste oluşturduk
dokumanlar = []

# Veri setimizdeki 'train' bölümündeki her bir satırı (şarkıyı) dolaşalım
# Not: veri_seti['train'] yerine veri_seti['test'] de kullanabilirsiniz
for satir in veri_seti["train"]:
    # 1. 'Lyrics' sütununu ana içerik (page_content) olarak al
    icerik = satir["Lyrics"]

    # 2. 'Artist' ve 'Song' sütunlarını etiket (metadata) olarak al
    etiketler = {
        "artist": satir["Artist"],
        "song": satir["Song"]
    }


    yeni_dokuman = Document(page_content=icerik, metadata=etiketler)

    # listeye ekledik
    dokumanlar.append(yeni_dokuman)

print(f"Toplam {len(dokumanlar)} adet şarkı dokümanı hazırlandı.")
print("\nİşte ilk dokümandan bir örnek:")
print(dokumanlar[0])

Veri seti 'Doküman' formatına dönüştürülüyor...
Toplam 13596 adet şarkı dokümanı hazırlandı.

İşte ilk dokümandan bir örnek:
page_content='80 contributorstranslationsenglishcivil war lyrics
what we've got here is failure to communicate
some men  you just can't reach
so you get what we had here last week
which is the way he wants it' metadata={'artist': 'guns n roses', 'song': 'civil war'}


In [ ]:
from langchain_community.vectorstores import Chroma

#chroma veri tabanı oluşturduk

print("Chroma veritabanı oluşturuluyor... (Bu işlem biraz sürebilir)")

vector_store = Chroma.from_documents(
    documents=dokumanlar,     # doküman listesi
    embedding=embeddings      # hugging Face modeli
)

print("Veritabanı (vector_store) başarıyla oluşturuldu!")

#test ettim
print("\n--- Veritabanı Testi ---")
test_sozu = "I'm waking up to ash and dust"

# veritabanında bu söze en çok benzeyen şarkıyı buluyoruz
benzer_dokumanlar = vector_store.similarity_search(test_sozu, k=1)

print(f"\n'{test_sozu}' Girdiğiniz şarkı sözüne en çok benzeyen şarkı bulundu.")
print("İşte o şarkılar:")
# bulunan dokümanın 'metadatasını yazdırıyoruz.
print(benzer_dokumanlar[0].metadata)

Chroma veritabanı oluşturuluyor... (Bu işlem biraz sürebilir)


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:

# en yakın 3 şarkıyı bulacak
retriever_multiple = vector_store.as_retriever(search_kwargs={"k": 3})

print("Retriever (Getirici) 3 sonuç getirecek şekilde ayarlandı.")


# düzgün bir metin listesine dönüştürür.
def format_metadata_list(docs):
    if not docs:
        return "Üzgünüm, bu söze benzer bir şarkı bulamadım."

    formatted_list = []
    for i, doc in enumerate(docs):
        # metadata'nın var olup olmadığını kontrol edelim
        if doc.metadata:
            artist = doc.metadata.get('artist', 'Bilinmeyen Sanatçı')
            song = doc.metadata.get('song', 'Bilinmeyen Şarkı')
            formatted_list.append(f"  {i+1}. Sanatçı: {artist}, Şarkı: {song}")
        else:
            formatted_list.append(f"  {i+1}. Metadata bulunamadı.")

    return "\n".join(formatted_list)

# 1. Prompt Şablonunu Güncelle
# LLM'e (Gemini'ye) artık bir LİSTE verdiğimizi ve bunu sunmasını söylüyoruz.
template_multiple = """
Kullanıcı bir şarkı sözü verdi. Veritabanından bu söze en çok benzeyen şarkıları buldum.
Şimdi bu bilgiyi kullanarak kullanıcıya şarkının adını ve sanatçısını söyle.
Cevabın kısa, net ve samimi bir dilde olsun.
Bulunan Olası Şarkılar (Context):
{context}

Kullanıcının Girdiği Söz (Question):
{question}

Cevabın (Türkçe):
"""
prompt_multiple = PromptTemplate.from_template(template_multiple)

# rag chain oluşturuyoruz
rag_chain_multiple = (
    {"context": retriever_multiple | format_metadata_list, "question": RunnablePassthrough()}
    | prompt_multiple
    | llm
    | StrOutputParser()
)
#kontrol
print("\nBirden fazla sonuç gösteren RAG Zinciri (rag_chain_multiple) başarıyla oluşturuldu!")

In [ ]:
print("CHATBOT TESTİ")

sarki_sozu_1 = "I'm waking up to ash and dust"
# HATA BURADAYDI: 'rag_chain' yerine 'rag_chain_multiple' olmalı
cevap_1 = rag_chain_multiple.invoke(sarki_sozu_1)

print(f"Soru: {sarki_sozu_1}")
print(f"Cevap: {cevap_1}")

In [ ]:
!pip install gradio -q
import gradio as gr

# --- 1. CHATBOT FONKSİYONUMUZU TANIMLAYALIM ---
# Gradio arayüzü, bu fonksiyonu çağıracak.

def find_song_from_lyrics(lyric_snippet):
    """
    Kullanıcıdan gelen şarkı sözü parçasını alır ve
    'rag_chain_multiple' zincirini kullanarak şarkıyı arar.
    """
    try:
        # Daha önce "Bölüm 1"de oluşturduğumuz 'çoklu sonuç' zincirini çağır
        response = rag_chain_multiple.invoke(lyric_snippet)
        return response
    except NameError:
        return ("HATA: 'rag_chain_multiple' değişkeni tanımlanmamış."
                "Lütfen bu hücreden önceki 'Bölüm 1' kodlarını çalıştırdığınızdan emin olun.")
    except Exception as e:
        return f"Beklenmedik bir hata oluştu: {str(e)}"

# --- 2. GRADIO ARAYÜZÜNÜ TASARLAYALIM ---

demo = gr.Interface(
    fn=find_song_from_lyrics,
    inputs=gr.Textbox(
        lines=5,  # Metin kutusunun yüksekliği (5 satır)
        label="Şarkı Sözü Parçası:",
        placeholder="Aklınıza takılan şarkı sözünü buraya yazın...\n\nörn: I'm waking up to ash and dust"
    ),
    outputs=gr.Textbox(
        label="Bulunan Olası Şarkılar:",
        lines=8  # Cevap kutusunun yüksekliği (8 satır)
    ),
    title="🎵 Şarkı Sözünden Şarkı Bulma Chatbot'u 🎵",
    description="Aklınıza takılan bir şarkı sözü parçasını yazın. RAG ve Gemini kullanarak veritabanımdaki 13,000+ şarkı arasından en benzer 3 olasılığı size listeleyeyim.",

    # Kullanıcıların denemesi için örnekler
    examples=[
        ["I'm waking up to ash and dust"],
        ["Yesterday, all my troubles seemed so far away"],
        ["I've become so numb, I can't feel you there"]
    ],

    # Arayüz teması (daha modern bir görünüm için)
    theme=gr.themes.Soft()
)

# --- 3. ARAYÜZÜ BAŞLATALIM ---

# Arayüzü başlat ve paylaşılabilir bir link (share=True) oluştur
print("Gradio arayüzü başlatılıyor...")
demo.launch(share=True, debug=True)